# Matched movement steering: linear probe versus sparse transcoder

This experiment asks whether handles discovered from contrasting activations can **cause a requested change in actual robot movement**. The available checkpoint is a timestep-conditioned transcoder, not a sparse autoencoder. We compare a dense linear-probe direction with a small set of native sparse features at the same action-expert MLP output.

1. **Opposite examples:** on identical images and initial simulator states, instruct Pi0.5 to pick the goal bowl or the distractor bowl. Both instructions come from LIBERO descriptions of the actual bowl locations. Record actual gripper movement and matched dense/sparse activations. Keep training pairs only if their native movement contrast exceeds the prespecified threshold; never filter validation/test by their outcomes.
2. **Discovery:** fit a standardized dense linear probe; rank sparse features by training-only standardized activation contrast and fit a sparse readout on the selected units. Pool action tokens and denoising calls identically. Validation AUC measures prediction, not causal importance.
3. **Validation:** intervene in both directions at candidate layers. Choose ONE shared layer by the mean movement effect of both arms. Freeze layer, features, directions, dose and test IDs before opening test cases.
4. **Held-out test:** execute the first 50 predicted actions in the simulator, with zero, half/full dose, random, budget-capped ablation and random-ablation interventions. Every condition constructs a fresh simulator with its seed set before construction, bypasses LeRobot’s success-triggered auto-reset for a fixed measurement horizon, uses identical starting pixels and noise, and preserves the native MLP plus an intervention delta. Sparse latent edits remain nonnegative. Both methods have the same per-token maximum output-RMS budget; report realized norms because sparse constraints may limit spending.
5. **Primary score:** let `P = (distance_to_distractor - distance_to_target) / bowl_separation`, using fixed initial bowl positions and the final measured gripper position. Score = `50 × desired_sign × (P_intervened − P_baseline)`. Report **sparse score − dense score**, averaging both directions, weighting tasks equally, with a paired task-bootstrap interval when at least three test tasks exist. Also report measured XYZ displacement, force/lift proxies relative to the original task, and whether capped ablation reduces the native prompt contrast. A distractor-directed instruction may intentionally lift that bowl, so the original-task harm proxy is not an intent-aware safety label.

Full split: tasks 0–4 train (10 starts each), 5–6 validation (3 each), 7–9 test (5 each). Both prompt variants and every intervention stay in the same task split. Candidate layers default to 0, 8, 17. Full defaults require about **574 single-chunk simulator trials**; smoke requires 86 and is only an infrastructure check. There is no 15-minute cutoff that prevents reaching the test stage. Each completed trial is resumable. Initial state arrays and camera pixels are checked before inference and saved per case; any mismatch writes reset diagnostics and stops before policy work. The reset-v2 protocol requires a new output folder to keep older reused-simulator trials separate. Interrupting the cell keeps completed trials; rerun with identical controls to resume.

These tests establish limited causal evidence for a handle, not proof of a uniquely represented concept, long-horizon task control, or safety. One fixed random control is not a random-control significance test. The existing policy/transcoder may have seen test tasks during pretraining; the split holds out handle discovery and selection. Do not tune settings after inspecting test scores and reuse the same test set as if untouched.

Setup commands install packages and prepare caches. **All experiment definitions are in the notebook; discovery, fitting and simulation execute directly in this kernel with native tracebacks.** No Python worker runs the experiment. Use a new run name after changing the protocol. Keep the existing occlusion experiment on its original branch.


In [ ]:
# @title Controls
DRIVE_ROOT = "/content/drive/MyDrive/groot-run-shared-programmer908" # @param {type:"string"}
DRIVE_FOLDER_ID = "13z_Qh91Ww0jdXoju2kQsG12AaEPKZsnH"
AUTO_CREATE_DRIVE_SHORTCUT = True
APPROVED_ACCOUNTS = "programmer908@gmail.com"
REPO_URL = "https://github.com/MarquiseRosier/pi05-run.git"
REPO_BRANCH = "feature/marquise-matched-steering"
REQUIRED_GPU = "Auto" # @param ["Auto", "A100", "L4"]
MIN_GPU_MEMORY_GB = 22
MIN_SYSTEM_RAM_GB = 24
CACHE_TRANSFER_MODE = "archive"
ALLOW_AUTH_REFRESH = True
FORCE_AUTH_REFRESH = False
HF_OFFLINE = True
INFERENCE_BATCH_SIZE = 1
RUN_NAME = "matched-steering-reset-002" # @param {type:"string"}
STUDY_SIZE = "full" # @param ["smoke", "full"]
PHASE = "all" # @param ["discovery", "validation", "test", "all"]
CANDIDATE_LAYERS = "0,8,17" # @param {type:"string"}
SPARSE_FEATURES = 16 # @param {type:"integer"}
RMS_BUDGET = 0.1 # @param {type:"number"}
CHECKPOINT = "" # @param {type:"string"}


In [ ]:
# @title Mount Drive And Validate Runtime

from pathlib import Path
import os
import subprocess
import time

from google.colab import drive


def parse_gib_from_meminfo() -> int:
    with open("/proc/meminfo", "r", encoding="utf-8") as handle:
        for line in handle:
            if line.startswith("MemTotal:"):
                kb = int(line.split()[1])
                return (kb + 1024 * 1024 - 1) // (1024 * 1024)
    return 0


def validate_colab_runtime():
    import shutil
    gpu_help = ("NVIDIA GPU tools are unavailable. In Colab choose Runtime -> Change runtime type -> "
                "A100 or L4 GPU, save, reconnect, and rerun from Controls. A CPU/TPU session cannot run this experiment.")
    smi = shutil.which("nvidia-smi")
    if not smi:
        smi = next((p for p in ("/usr/bin/nvidia-smi", "/usr/local/nvidia/bin/nvidia-smi")
                    if Path(p).is_file() and os.access(p, os.X_OK)), None)
    if not smi:
        raise RuntimeError(gpu_help)
    result = subprocess.run(
        [smi, "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader,nounits"],
        text=True,
        capture_output=True,
    )
    print(result.stdout or result.stderr)
    if result.returncode != 0 or not result.stdout.strip():
        raise RuntimeError(gpu_help)

    gpu_line = result.stdout.strip().splitlines()[0]
    parts = [part.strip() for part in gpu_line.split(",")]
    gpu_name = parts[0]
    gpu_mem_gb = int((int(parts[2]) + 1023) // 1024) if len(parts) >= 3 and parts[2].isdigit() else 0
    system_ram_gb = parse_gib_from_meminfo()
    print(f"Runtime memory: GPU={gpu_name} ~{gpu_mem_gb} GiB VRAM | system RAM ~{system_ram_gb} GiB")

    if REQUIRED_GPU == "Auto" and not any(model in gpu_name.upper() for model in ("A100", "L4")):
        raise RuntimeError(f"Auto mode supports A100 or L4; allocated {gpu_name}. Change the Colab runtime.")
    if REQUIRED_GPU not in ("Any", "Auto") and REQUIRED_GPU.lower() not in gpu_name.lower():
        raise RuntimeError(
            f"Requested {REQUIRED_GPU}, but Colab allocated {gpu_name}. "
            "Use Runtime -> Change runtime type -> GPU -> L4/A100, then rerun from this cell."
        )
    if int(MIN_GPU_MEMORY_GB) > 0 and gpu_mem_gb < int(MIN_GPU_MEMORY_GB):
        raise RuntimeError(
            f"GPU VRAM is too small for Pi0.5 LIBERO: {gpu_name} has ~{gpu_mem_gb} GiB, "
            f"need >= {MIN_GPU_MEMORY_GB} GiB. T4 usually exits 137 while loading the policy. "
            "Switch to L4 or A100 before running eval."
        )
    if int(MIN_SYSTEM_RAM_GB) > 0 and system_ram_gb < int(MIN_SYSTEM_RAM_GB):
        raise RuntimeError(
            f"System RAM is too small for Pi0.5 LIBERO: runtime has ~{system_ram_gb} GiB, "
            f"need >= {MIN_SYSTEM_RAM_GB} GiB. In Colab, choose a high-RAM L4/A100 runtime."
        )

    requested_batch = int(globals().get("INFERENCE_BATCH_SIZE", 0))
    if requested_batch < 0:
        raise ValueError("INFERENCE_BATCH_SIZE must be >= 0")
    batch = requested_batch or (1 if "L4" in gpu_name.upper() else 4)
    print(f"Inference batch size: {batch} (0 in Controls means automatic)")
    return {"gpu_name": gpu_name, "gpu_memory_gib": gpu_mem_gb, "batch_size": batch}


RUNTIME_PROFILE = validate_colab_runtime()
EFFECTIVE_BATCH_SIZE = RUNTIME_PROFILE["batch_size"]


def mount_drive_with_retry(mountpoint: str = "/content/drive", attempts: int = 3) -> None:
    """Mount Drive, retrying because `mount failed` is usually transient.

    Colab raises a bare ``ValueError: mount failed`` that says nothing about the
    cause. By far the most common one here is other live Colab sessions each
    holding their own Drive mount -- easy to hit when the original/probe and
    replace notebooks are open at the same time.
    """
    if Path(mountpoint, "MyDrive").exists():
        print(f"Drive already mounted at {mountpoint}", flush=True)
        return

    last_error: Exception | None = None
    for attempt in range(1, attempts + 1):
        try:
            # force_remount clears a half-established mount left by a failed try.
            drive.mount(mountpoint, force_remount=attempt > 1)
            print(f"Drive mounted at {mountpoint} (attempt {attempt})", flush=True)
            return
        except Exception as exc:  # Colab raises a bare ValueError here.
            last_error = exc
            print(f"Drive mount attempt {attempt}/{attempts} failed: {exc}", flush=True)
            if attempt < attempts:
                delay = 5 * attempt
                print(f"  retrying in {delay}s", flush=True)
                time.sleep(delay)

    raise RuntimeError(
        f"Could not mount Google Drive after {attempts} attempts (last error: {last_error}).\n"
        "This notebook needs Drive: the transcoder checkpoint, HF cache and outputs all live there.\n"
        "Most likely causes, in order:\n"
        "  1. Other Colab sessions are holding Drive mounts. Runtime > Manage sessions, "
        "terminate the ones you are not using, then Runtime > Restart session and rerun.\n"
        "  2. The authorization popup was blocked or dismissed. Allow popups and third-party "
        "cookies for colab.research.google.com, then rerun.\n"
        "  3. A transient Drive outage. Restart the runtime and try again in a few minutes."
    ) from last_error


mount_drive_with_retry()


def ensure_drive_root_visible(root_value: str) -> Path:
    root = Path(root_value)
    if root.exists():
        return root

    folder_id = str(globals().get("DRIVE_FOLDER_ID", "")).strip()
    auto_shortcut = bool(globals().get("AUTO_CREATE_DRIVE_SHORTCUT", True))
    mydrive_prefixes = ("/content/drive/MyDrive/", "/content/drive/My Drive/")
    if folder_id and auto_shortcut and str(root).startswith(mydrive_prefixes):
        print("Drive root is not visible yet; creating a My Drive shortcut to the shared folder.", flush=True)
        try:
            from google.colab import auth
            auth.authenticate_user()
            try:
                from googleapiclient.discovery import build
            except Exception:
                subprocess.run(["python3", "-m", "pip", "install", "-q", "-U", "google-api-python-client"], check=True)
                from googleapiclient.discovery import build
            service = build("drive", "v3")
            metadata = {
                "name": root.name,
                "mimeType": "application/vnd.google-apps.shortcut",
                "shortcutDetails": {"targetId": folder_id},
                "parents": ["root"],
            }
            created = service.files().create(body=metadata, fields="id,name").execute()
            print(f"Created Drive shortcut: {created.get('name')} ({created.get('id')})", flush=True)
            for _ in range(12):
                if root.exists():
                    return root
                time.sleep(5)
        except Exception as exc:
            raise RuntimeError(
                "Could not auto-create the Drive shortcut. Confirm the folder was shared with this Google account "
                "and DRIVE_FOLDER_ID is the shared folder ID."
            ) from exc
        if root.exists():
            return root
        raise RuntimeError(
            f"Drive shortcut was created, but {root} is not visible in the mounted filesystem yet. "
            "Rerun this cell, or use a Google Workspace Shared Drive path."
        )

    if folder_id and str(root).startswith("/content/drive/Shareddrives/"):
        raise RuntimeError(
            f"Shared Drive path is not visible: {root}. Confirm the teammate has access to the Shared Drive."
        )

    if not folder_id:
        print(
            "DRIVE_FOLDER_ID is blank and DRIVE_ROOT does not exist. Creating a new folder at DRIVE_ROOT; "
            "this is intended only for initial owner setup. For zero teammate setup, set DRIVE_FOLDER_ID in the notebook.",
            flush=True,
        )
    return root


DRIVE_ROOT = ensure_drive_root_visible(DRIVE_ROOT)
DRIVE_ARCHIVES = DRIVE_ROOT / "archives"
DRIVE_HF_HOME = DRIVE_ROOT / "hf_home"
DRIVE_LIBERO_CACHE = DRIVE_ROOT / "libero_cache"
DRIVE_LIBERO_DATASETS = DRIVE_ROOT / "libero_datasets"
DRIVE_OUTPUTS = DRIVE_ROOT / "outputs"
DRIVE_NOTEBOOK_META = DRIVE_ROOT / "notebook_meta"
for path in [DRIVE_ROOT, DRIVE_ARCHIVES, DRIVE_HF_HOME, DRIVE_LIBERO_CACHE, DRIVE_LIBERO_DATASETS, DRIVE_OUTPUTS, DRIVE_NOTEBOOK_META]:
    path.mkdir(parents=True, exist_ok=True)

HF_HOME_ARCHIVE = DRIVE_ARCHIVES / "hf_home.tar"
LIBERO_CACHE_ARCHIVE = DRIVE_ARCHIVES / "libero_cache.tar"
LIBERO_DATASETS_ARCHIVE = DRIVE_ARCHIVES / "libero_datasets.tar"

print("Drive root:", DRIVE_ROOT)
print("Archive dir:", DRIVE_ARCHIVES)
print("Expected sharing: restricted to", APPROVED_ACCOUNTS)


In [ ]:
# @title Install Pinned LeRobot Runtime (Repairs Existing Environment)

import os
import sys
import subprocess
from pathlib import Path

# Set cache and rendering variables before importing Transformers or LeRobot;
# these libraries cache environment settings at import time in the kernel.
os.environ.update({
    "HF_HOME": "/content/hf_home",
    "HF_HUB_CACHE": "/content/hf_home/hub",
    "MUJOCO_GL": "egl",
    "PYOPENGL_PLATFORM": "egl",
    "MUJOCO_EGL_DEVICE_ID": "0",
    "MPLBACKEND": "Agg",
    "LIBERO_CONFIG_PATH": "/content/groot-run/.libero",
    "LIBERO_DATASET_DIR": "/content/groot-run/data/libero/datasets",
})
for key in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"):
    if HF_OFFLINE:
        os.environ[key] = "1"
    else:
        os.environ.pop(key, None)
loaded_hub = sys.modules.get("huggingface_hub.constants")
if loaded_hub is not None and loaded_hub.HF_HUB_CACHE != os.environ["HF_HUB_CACHE"]:
    raise RuntimeError("Restart the Colab session, then rerun from the top to use the configured model cache.")

PYTHON = Path(sys.executable)  # Install into the kernel used for direct collection.
UV_BIN_DIR = Path("/content/uv-bin")
UV = str(UV_BIN_DIR / "uv")


def run(cmd, *, env=None):
    # External setup tools (uv/apt/git), never a Python collection worker.
    # Stream stderr too: inherited child output may be absent from Colab's cell.
    from collections import deque
    import shlex
    cmd = list(map(str, cmd))
    print("$", shlex.join(cmd), flush=True)
    tail = deque(maxlen=100)
    with subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                tail.append(line)
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if returncode:
        detail = "".join(tail).strip() or "The command produced no output."
        raise RuntimeError(f"Setup command failed (exit {returncode}): {shlex.join(cmd)}\n\n{detail}")
    return subprocess.CompletedProcess(cmd, returncode)


def install_uv() -> str:
    # Colab may already have /usr/local/bin/uv, but its version/behavior can differ.
    # Use a notebook-owned binary so every clean runtime follows the same path.
    UV_BIN_DIR.mkdir(parents=True, exist_ok=True)
    installer = Path("/tmp/install-uv.sh")
    run(["curl", "-LsSf", "https://astral.sh/uv/install.sh", "-o", installer])
    env = os.environ.copy()
    env["UV_INSTALL_DIR"] = str(UV_BIN_DIR)
    run(["sh", installer], env=env)
    os.environ["PATH"] = f"{UV_BIN_DIR}:" + os.environ["PATH"]
    run([UV, "--version"])
    return UV


# Check in this kernel; do not mix a Python 3.12 venv into a Python 3.13 kernel.
# Package installation cannot replace native modules already loaded by this process.
loaded_runtime = any(name in sys.modules for name in ("torch", "numpy", "transformers", "lerobot"))
try:
    import importlib.util
    from importlib.metadata import version
    assert version('lerobot') == '0.4.4'
    from transformers.models.siglip import check
    assert check.check_whether_transformers_replace_is_installed_correctly()
    import torch
    assert torch.cuda.is_available()
    assert all(importlib.util.find_spec(name) for name in ('libero', 'langfuse', 'sklearn', 'joblib', 'PIL'))
    ready = True
except (ImportError, AssertionError, ValueError, RuntimeError) as exc:
    print("Runtime needs installation:", exc)
    ready = False
if ready:
    print("Reusing existing CUDA/LeRobot runtime:", PYTHON)
    if not Path(UV).exists():
        UV = install_uv()
else:
    apt_packages = [
        "build-essential", "ca-certificates", "cmake", "curl", "ffmpeg", "git", "pv", "rsync",
        "libegl1", "libgl1", "libglib2.0-0", "libglvnd0", "libglx0", "libopengl0",
        "libosmesa6-dev", "libsm6", "libxext6", "libxrender1", "pkg-config",
    ]
    apt_env = os.environ.copy()
    apt_env["DEBIAN_FRONTEND"] = "noninteractive"
    run(["apt-get", "update", "-qq"], env=apt_env)
    run(["apt-get", "install", "-y", "-qq", *apt_packages], env=apt_env)
    UV = install_uv()

    # egl-probe's source build predates CMake 4. Install its build tools first.
    run([UV, "pip", "install", "--python", str(PYTHON), "--system", "--break-system-packages",
         "cmake==3.31.6", "setuptools>=71,<81", "wheel"])
    import sysconfig
    os.environ["PATH"] = sysconfig.get_path("scripts") + os.pathsep + os.environ["PATH"]
    run(["cmake", "--version"])
    run([UV, "pip", "install", "--python", str(PYTHON), "--system", "--break-system-packages",
         "--no-build-isolation-package", "egl-probe", "egl-probe==1.0.2"])

    # Match scripts/pi05_transformers_overrides.txt; the repo is cloned in the next cell.
    transformers_override = Path("/content/pi05-transformers-overrides.txt")
    transformers_override.write_text('transformers @ git+https://github.com/huggingface/transformers.git@dcddb970176382c0fcf4521b0c0e6fc15894dfe0' + "\n")

    run([
        UV, "pip", "install", "--python", str(PYTHON), "--system", "--break-system-packages",
        "--torch-backend", "cu128",
        "--overrides", str(transformers_override),
        "--no-build-isolation-package", "egl-probe", "cmake==3.31.6",
        "lerobot[libero]==0.4.4", "hf-transfer", "opencv-python", "numpy", "langfuse",
        "scikit-learn", "joblib", "pillow",
    ])

    print("Installed into notebook Python:", PYTHON)
    # The readiness probe itself may also have imported modules before failing.
    if loaded_runtime or any(name in sys.modules for name in ("torch", "numpy", "transformers", "lerobot")):
        raise RuntimeError("Packages installed. Restart the Colab session once, then rerun the notebook. "
                           "This clears old Python/native modules; Drive caches are preserved.")


In [ ]:
# @title Clone Or Update Repo

from pathlib import Path
import subprocess

LOCAL_REPO = Path("/content/groot-run")
if LOCAL_REPO.exists():
    subprocess.run(["git", "-C", str(LOCAL_REPO), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(LOCAL_REPO)], check=True)

print("Repo:", LOCAL_REPO)
subprocess.run(["git", "-C", str(LOCAL_REPO), "rev-parse", "--short", "HEAD"], check=True)
# Validate the actual notebook interpreter before restoring large caches.
import sys
sys.path[:0] = [str(LOCAL_REPO / "scripts"), str(LOCAL_REPO / "src")]
os.environ["MPLBACKEND"] = "Agg"
from pi05_monitor_runtime import validate_runtime
print(validate_runtime(require_cuda=True))


In [ ]:
# @title Prepare Drive Caches And Offline Mode

from pathlib import Path
import os
import shutil
import shlex
import subprocess
import time

LOCAL_HF_HOME = Path("/content/hf_home")
LOCAL_LIBERO_CACHE = Path.home() / ".cache/libero"
LOCAL_OUTPUT_ROOT = LOCAL_REPO / "outputs/eval/pi05_libero"
LOCAL_DATA_ROOT = LOCAL_REPO / "data/libero/datasets"
LOCAL_LIBERO_CONFIG = LOCAL_REPO / ".libero"
for path in [LOCAL_OUTPUT_ROOT, LOCAL_LIBERO_CONFIG]:
    path.mkdir(parents=True, exist_ok=True)

HF_CACHE_REFRESHED = False

repos = ["lerobot/pi05_libero_finetuned", "google/paligemma-3b-pt-224", "lerobot/libero-assets"]


def hf_repo_type(repo_id: str) -> str:
    return "dataset" if repo_id == "lerobot/libero-assets" else "model"


def now_text() -> str:
    return time.strftime("%H:%M:%S")


def format_duration(seconds: float | None) -> str:
    if seconds is None or seconds != seconds or seconds < 0:
        return "unknown"
    seconds = int(seconds)
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}h{minutes:02d}m{secs:02d}s"
    if minutes:
        return f"{minutes}m{secs:02d}s"
    return f"{secs}s"


def format_bytes(value: float | int | None) -> str:
    if value is None:
        return "unknown"
    value = float(value)
    units = ["B", "KiB", "MiB", "GiB", "TiB"]
    for unit in units:
        if abs(value) < 1024 or unit == units[-1]:
            return f"{value:.1f}{unit}" if unit != "B" else f"{value:.0f}{unit}"
        value /= 1024
    return f"{value:.1f}PiB"


def timed(label, fn):
    print(f"\n== {label} | start {now_text()} ==", flush=True)
    start = time.time()
    result = fn()
    elapsed = time.time() - start
    print(f"== done: {label} | elapsed {format_duration(elapsed)} | finish {now_text()} ==", flush=True)
    return result


def run(cmd, **kwargs):
    print("+", " ".join(map(str, cmd)), flush=True)
    start = time.time()
    result = subprocess.run([str(x) for x in cmd], check=True, **kwargs)
    print(f"+ done in {format_duration(time.time() - start)}", flush=True)
    return result


def reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def dir_has_anything(path: Path) -> bool:
    return path.exists() and any(path.iterdir())


def hf_cache_has(repo_id: str, root: Path) -> bool:
    namespace, name = repo_id.split("/", 1)
    prefix = "datasets" if hf_repo_type(repo_id) == "dataset" else "models"
    return (root / "hub" / f"{prefix}--{namespace}--{name}").exists()


def get_hf_token(required: bool = False) -> str:
    token = os.environ.get("HF_TOKEN", "").strip()
    if token:
        print("Using HF_TOKEN from the environment.", flush=True)
        return token
    try:
        from google.colab import userdata
        token = (userdata.get("HF_TOKEN") or "").strip()
        if token:
            os.environ["HF_TOKEN"] = token
            print("Using private Colab Secret HF_TOKEN.", flush=True)
            return token
    except Exception:
        pass
    if required:
        raise RuntimeError(
            "Set the HF_TOKEN environment variable or enable notebook access to the HF_TOKEN Colab Secret."
        )
    return ""


def path_size_bytes(path: Path) -> int:
    if not path.exists():
        return 0
    total = 0
    for root, dirs, files in os.walk(path):
        dirs[:] = [name for name in dirs if not Path(root, name).is_symlink()]
        for name in files:
            file_path = Path(root, name)
            try:
                total += file_path.lstat().st_size
            except OSError:
                pass
    return total


def du(path: Path):
    if path.exists():
        run(["du", "-sh", path])


def extract_tar(archive: Path, dst: Path):
    reset_dir(dst)
    size = archive.stat().st_size
    print(f"Archive: {archive} ({format_bytes(size)}) -> {dst}", flush=True)
    cmd = f"set -euo pipefail; pv -ptebarf -s {size} {shlex.quote(str(archive))} | tar -C {shlex.quote(str(dst))} -xf -"
    run(["bash", "-lc", cmd])
    du(dst)


def create_tar(src: Path, archive: Path):
    if not dir_has_anything(src):
        print(f"Skipping archive for empty directory: {src}", flush=True)
        return
    archive.parent.mkdir(parents=True, exist_ok=True)
    tmp = archive.with_suffix(archive.suffix + ".tmp")
    if tmp.exists():
        tmp.unlink()
    size_bytes = path_size_bytes(src)
    print(f"Archiving: {src} ({format_bytes(size_bytes)}) -> {archive}", flush=True)
    cmd = f"set -euo pipefail; tar -C {shlex.quote(str(src))} -cf - . | pv -ptebarf -s {size_bytes} > {shlex.quote(str(tmp))}"
    run(["bash", "-lc", cmd])
    tmp.replace(archive)
    run(["ls", "-lh", archive])


def rsync_tree(src: Path, dst: Path, reset: bool = True):
    src.mkdir(parents=True, exist_ok=True)
    if reset:
        reset_dir(dst)
    else:
        dst.mkdir(parents=True, exist_ok=True)
    print(f"Rsync: {src} ({format_bytes(path_size_bytes(src))}) -> {dst}", flush=True)
    run(["rsync", "-a", "--human-readable", "--info=progress2", "--stats", f"{src}/", f"{dst}/"])
    du(dst)


if CACHE_TRANSFER_MODE == "archive" and HF_HOME_ARCHIVE.exists() and not FORCE_AUTH_REFRESH:
    timed("extract HF cache archive from Drive to /content", lambda: extract_tar(HF_HOME_ARCHIVE, LOCAL_HF_HOME))
elif ALLOW_AUTH_REFRESH:
    token = get_hf_token(required=True)
    reset_dir(LOCAL_HF_HOME)
    run([str(PYTHON), "-c", "import huggingface_hub; print('huggingface_hub OK')"])
    refresh_code = """
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.utils import enable_progress_bars
import os
import subprocess
import threading
import time

repos = os.environ['REFRESH_REPOS'].split(',')
token = os.environ.get('HF_TOKEN') or None
cache_dir = os.environ['LOCAL_HF_HUB_CACHE']


def repo_type(repo):
    return 'dataset' if repo == 'lerobot/libero-assets' else 'model'
heartbeat_seconds = int(os.environ.get('HF_PROGRESS_HEARTBEAT_SECONDS', '15'))


def now_text():
    return time.strftime('%H:%M:%S')


def format_duration(seconds):
    if seconds is None or seconds != seconds or seconds < 0:
        return 'unknown'
    seconds = int(seconds)
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f'{hours}h{minutes:02d}m{secs:02d}s'
    if minutes:
        return f'{minutes}m{secs:02d}s'
    return f'{secs}s'


def format_bytes(value):
    if value is None:
        return 'unknown'
    value = float(value)
    units = ['B', 'KiB', 'MiB', 'GiB', 'TiB']
    for unit in units:
        if abs(value) < 1024 or unit == units[-1]:
            return f'{value:.1f}{unit}' if unit != 'B' else f'{value:.0f}{unit}'
        value /= 1024
    return f'{value:.1f}PiB'


def path_size_bytes(path):
    if not os.path.exists(path):
        return 0
    total = 0
    for root, dirs, files in os.walk(path):
        dirs[:] = [name for name in dirs if not os.path.islink(os.path.join(root, name))]
        for name in files:
            file_path = os.path.join(root, name)
            try:
                total += os.lstat(file_path).st_size
            except OSError:
                pass
    return total


def estimate_repo_size(api, repo):
    try:
        if repo_type(repo) == 'dataset':
            info = api.dataset_info(repo_id=repo, token=token, files_metadata=True)
        else:
            info = api.model_info(repo_id=repo, token=token, files_metadata=True)
        sizes = [getattr(sibling, 'size', None) for sibling in getattr(info, 'siblings', [])]
        total = sum(size for size in sizes if isinstance(size, int))
        count = len(sizes)
        return count, total or None
    except Exception as exc:
        print(f'[{repo}] could not estimate size before download: {type(exc).__name__}: {exc}', flush=True)
        return None, None


def heartbeat(repo, root, start_bytes, estimated_total, stop_event):
    last_time = time.time()
    last_bytes = start_bytes
    start_time = last_time
    while not stop_event.wait(heartbeat_seconds):
        now = time.time()
        current_bytes = path_size_bytes(root)
        delta = max(0, current_bytes - start_bytes)
        recent_rate = max(0, current_bytes - last_bytes) / max(0.001, now - last_time)
        avg_rate = delta / max(0.001, now - start_time)
        if estimated_total and avg_rate > 0:
            remaining = max(0, estimated_total - delta)
            eta = remaining / avg_rate
            pct = min(100.0, (delta / estimated_total) * 100.0)
            estimate_text = f'{pct:5.1f}% of est. {format_bytes(estimated_total)} | ETA {format_duration(eta)}'
        else:
            estimate_text = 'ETA unknown'
        print(
            f'[{repo}] progress {format_bytes(delta)} | recent {format_bytes(recent_rate)}/s | '
            f'avg {format_bytes(avg_rate)}/s | elapsed {format_duration(now - start_time)} | {estimate_text}',
            flush=True,
        )
        last_time = now
        last_bytes = current_bytes


os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '0'
os.environ['TQDM_DISABLE'] = '0'
os.environ['TQDM_MININTERVAL'] = '1'
try:
    enable_progress_bars()
except Exception as exc:
    print(f'Could not force-enable Hugging Face progress bars: {type(exc).__name__}: {exc}', flush=True)

api = HfApi()
for index, repo in enumerate(repos, 1):
    print(f'\\n[{index}/{len(repos)}] preparing {repo} at {now_text()}', flush=True)
    file_count, estimated_total = estimate_repo_size(api, repo)
    if file_count is not None:
        print(f'[{repo}] estimated remote files={file_count} size={format_bytes(estimated_total)}', flush=True)
    before = path_size_bytes(cache_dir)
    stop_event = threading.Event()
    monitor = threading.Thread(target=heartbeat, args=(repo, cache_dir, before, estimated_total, stop_event), daemon=True)
    start = time.time()
    monitor.start()
    try:
        path = snapshot_download(
            repo_id=repo,
            repo_type=repo_type(repo),
            cache_dir=cache_dir,
            token=token,
            max_workers=8,
        )
    finally:
        stop_event.set()
        monitor.join(timeout=2)
    elapsed = time.time() - start
    after = path_size_bytes(cache_dir)
    delta = max(0, after - before)
    avg_rate = delta / max(0.001, elapsed)
    print(f'[{repo}] -> {path}', flush=True)
    print(
        f'[{repo}] done at {now_text()} | elapsed {format_duration(elapsed)} | '
        f'cache delta {format_bytes(delta)} | avg {format_bytes(avg_rate)}/s | total cache {format_bytes(after)}',
        flush=True,
    )
"""
    env = os.environ.copy()
    for key in ["HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"]:
        env.pop(key, None)
    env.update({
        "PYTHONUNBUFFERED": "1",
        "HF_TOKEN": token,
        "HF_HOME": str(LOCAL_HF_HOME),
        "HF_HUB_CACHE": str(LOCAL_HF_HOME / "hub"),
        "LOCAL_HF_HUB_CACHE": str(LOCAL_HF_HOME / "hub"),
        "REFRESH_REPOS": ",".join(repos),
        "HF_HUB_ENABLE_HF_TRANSFER": "1",
        "HF_XET_HIGH_PERFORMANCE": "1",
        "HF_HUB_DISABLE_PROGRESS_BARS": "0",
        "TQDM_DISABLE": "0",
        "TQDM_MININTERVAL": "1",
        "HF_PROGRESS_HEARTBEAT_SECONDS": "15",
    })
    timed("download gated HF models to fast local disk", lambda: subprocess.run([str(PYTHON), "-u", "-c", refresh_code], env=env, check=True))
    HF_CACHE_REFRESHED = True
elif CACHE_TRANSFER_MODE == "folders" or dir_has_anything(DRIVE_HF_HOME):
    print("Archive missing; falling back to Drive folder rsync. This can be very slow for HF caches.", flush=True)
    timed("copy HF cache folder from Drive to /content", lambda: rsync_tree(DRIVE_HF_HOME, LOCAL_HF_HOME))
else:
    raise RuntimeError(
        "No HF cache archive or folder found in Drive. Set ALLOW_AUTH_REFRESH=True once with the HF_TOKEN environment variable or an enabled HF_TOKEN Colab Secret."
    )

missing_local = [repo for repo in repos if not hf_cache_has(repo, LOCAL_HF_HOME)]
if missing_local and ALLOW_AUTH_REFRESH:
    token = get_hf_token(required=False)
    refresh_missing_code = """
from huggingface_hub import snapshot_download
import os

repos = os.environ['MISSING_REPOS'].split(',')
token = os.environ.get('HF_TOKEN') or None
cache_dir = os.environ['LOCAL_HF_HUB_CACHE']


def repo_type(repo):
    return 'dataset' if repo == 'lerobot/libero-assets' else 'model'

for repo in repos:
    print('refreshing missing cache entry', repo, 'repo_type', repo_type(repo), flush=True)
    path = snapshot_download(
        repo_id=repo,
        repo_type=repo_type(repo),
        cache_dir=cache_dir,
        token=token,
        max_workers=8,
    )
    print(repo, '->', path, flush=True)
"""
    env = os.environ.copy()
    for key in ["HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"]:
        env.pop(key, None)
    env.update({
        "PYTHONUNBUFFERED": "1",
        "HF_TOKEN": token or "",
        "HF_HOME": str(LOCAL_HF_HOME),
        "HF_HUB_CACHE": str(LOCAL_HF_HOME / "hub"),
        "LOCAL_HF_HUB_CACHE": str(LOCAL_HF_HOME / "hub"),
        "MISSING_REPOS": ",".join(missing_local),
        "HF_HUB_ENABLE_HF_TRANSFER": "1",
        "HF_XET_HIGH_PERFORMANCE": "1",
        "HF_HUB_DISABLE_PROGRESS_BARS": "0",
        "TQDM_DISABLE": "0",
        "TQDM_MININTERVAL": "1",
    })
    timed("refresh missing HF cache entries", lambda: subprocess.run([str(PYTHON), "-u", "-c", refresh_missing_code], env=env, check=True))
    HF_CACHE_REFRESHED = True
    missing_local = [repo for repo in repos if not hf_cache_has(repo, LOCAL_HF_HOME)]

if missing_local:
    raise RuntimeError(
        "Local HF cache is missing: " + ", ".join(missing_local) + "\n"
        "Run once with ALLOW_AUTH_REFRESH=True plus the HF_TOKEN environment variable or an enabled HF_TOKEN Colab Secret, "
        "or populate DRIVE_ROOT/archives/hf_home.tar."
    )

if CACHE_TRANSFER_MODE == "archive" and LIBERO_CACHE_ARCHIVE.exists():
    timed("extract LIBERO cache archive from Drive", lambda: extract_tar(LIBERO_CACHE_ARCHIVE, LOCAL_LIBERO_CACHE))
else:
    timed("copy LIBERO cache folder from Drive", lambda: rsync_tree(DRIVE_LIBERO_CACHE, LOCAL_LIBERO_CACHE))

if CACHE_TRANSFER_MODE == "archive" and LIBERO_DATASETS_ARCHIVE.exists():
    timed("extract LIBERO datasets archive from Drive", lambda: extract_tar(LIBERO_DATASETS_ARCHIVE, LOCAL_DATA_ROOT))
else:
    timed("copy LIBERO datasets folder from Drive", lambda: rsync_tree(DRIVE_LIBERO_DATASETS, LOCAL_DATA_ROOT))

os.environ.update({
    "PATH": f"{PYTHON.parent}:" + os.environ["PATH"],
    "PYTHON": str(PYTHON),
    "HF_HOME": str(LOCAL_HF_HOME),
    "HF_HUB_CACHE": str(LOCAL_HF_HOME / "hub"),
    "HF_HUB_ENABLE_HF_TRANSFER": "1",
    "HF_XET_HIGH_PERFORMANCE": "1",
    "HF_HUB_DISABLE_PROGRESS_BARS": "0",
    "TQDM_DISABLE": "0",
    "TQDM_MININTERVAL": "1",
    "MUJOCO_GL": "egl",
    "PYOPENGL_PLATFORM": "egl",
    "MUJOCO_EGL_DEVICE_ID": "0",
    "MPLBACKEND": "Agg",
    "LIBERO_CONFIG_PATH": str(LOCAL_LIBERO_CONFIG),
    "LIBERO_DATASET_DIR": str(LOCAL_DATA_ROOT),
    "OUTPUT_ROOT": str(LOCAL_OUTPUT_ROOT),
})
if HF_OFFLINE:
    os.environ.update({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "HF_DATASETS_OFFLINE": "1"})
else:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)
    os.environ.pop("HF_DATASETS_OFFLINE", None)

validate_code = """
from huggingface_hub import snapshot_download
import os


def repo_type(repo):
    return 'dataset' if repo == 'lerobot/libero-assets' else 'model'

for repo in os.environ['VALIDATE_REPOS'].split(','):
    path = snapshot_download(repo_id=repo, repo_type=repo_type(repo), cache_dir=os.environ['HF_HUB_CACHE'], local_files_only=True)
    print(repo, '->', path, flush=True)
"""
env = os.environ.copy()
env["VALIDATE_REPOS"] = ",".join(repos)
timed("validate local HF cache with local_files_only=True", lambda: subprocess.run([str(PYTHON), "-u", "-c", validate_code], env=env, check=True))

install_assets_code = """
from pathlib import Path
import os
import shutil
import site

from huggingface_hub import snapshot_download

roots = [Path(path) for path in site.getsitepackages()]
user_site = site.getusersitepackages()
if user_site:
    roots.append(Path(user_site))
for root in roots:
    candidate = root / "libero" / "libero"
    if candidate.exists():
        libero_root = candidate
        break
else:
    raise SystemExit("Could not find installed libero package path")

assets_dir = libero_root / "assets"
required = assets_dir / "scenes" / "libero_tabletop_base_style.xml"
snapshot = Path(snapshot_download(
    repo_id="lerobot/libero-assets",
    repo_type="dataset",
    cache_dir=os.environ["HF_HUB_CACHE"],
    local_files_only=True,
))
if not required.exists():
    print(f"Installing LIBERO assets: {snapshot} -> {assets_dir}", flush=True)
    assets_dir.mkdir(parents=True, exist_ok=True)
    for child in snapshot.iterdir():
        if child.name == ".gitattributes":
            continue
        target = assets_dir / child.name
        if child.is_dir():
            shutil.copytree(child, target, dirs_exist_ok=True)
        else:
            shutil.copy2(child, target)
if not required.exists():
    raise SystemExit(f"LIBERO asset install failed; missing {required}")
print("libero_assets OK", required, flush=True)
"""
timed("install LIBERO assets into package", lambda: subprocess.run([str(PYTHON), "-u", "-c", install_assets_code], env=os.environ.copy(), check=True))

print("\nLocal cache summary:")
du(LOCAL_HF_HOME)
du(LOCAL_LIBERO_CACHE)
du(LOCAL_DATA_ROOT)


In [ ]:
# @title Check Analysis Dependencies In The Notebook Kernel
import sklearn
import joblib
from PIL import Image as PILImage
print("Analysis dependencies ready in the notebook kernel.")


In [ ]:
# @title Definitions: Task Identity And Contact Measurements
import re
import os
import json
import hashlib
from pathlib import Path
import numpy as np

def task_bddl_path(inner):
    """Use the active LIBERO domain, not a nonexistent LeRobot private field."""
    wrapper = getattr(inner, "_env", None)
    domain = getattr(wrapper, "env", None)
    for owner, attribute in ((domain, "bddl_file_name"), (wrapper, "bddl_file_name"),
                             (inner, "_task_bddl_file")):
        value = getattr(owner, attribute, None)
        if value:
            path = Path(value)
            if not path.is_file():
                raise FileNotFoundError(f"Active LIBERO task BDDL is missing: {path}")
            return path
    raise RuntimeError("Active LIBERO environment exposes no task BDDL path; "
                       "expected inner._env.env.bddl_file_name")

def bddl_sections(text):
    """Read balanced BDDL forms, including nested goals; ignore line comments."""
    stack, roots = [], []
    for token in re.findall(r"\(|\)|[^\s()]+", re.sub(r";[^\n]*", "", text)):
        if token == "(":
            form = []
            (stack[-1] if stack else roots).append(form)
            stack.append(form)
        elif token == ")":
            if not stack:
                raise ValueError("Unbalanced BDDL parentheses")
            stack.pop()
        elif stack:
            stack[-1].append(token)
        else:
            raise ValueError("Unexpected BDDL token outside a form")
    if stack or len(roots) != 1 or not roots[0] or roots[0][0] != "define":
        raise ValueError("Expected one balanced BDDL define form")
    sections = {}
    for form in roots[0][1:]:
        if isinstance(form, list) and form and str(form[0]).startswith(":"):
            if form[0] in sections:
                raise ValueError(f"Duplicate BDDL section: {form[0]}")
            sections[form[0]] = form[1:]
    return sections

def target_bowl_spec(inner):
    """Resolve the moved bowl from the goal, never from object enumeration order."""
    path = task_bddl_path(inner)
    contents = path.read_bytes()
    sections = bddl_sections(contents.decode())
    objects, pending = {}, []
    tokens = iter(sections.get(":objects", []))
    for token in tokens:
        if token == "-":
            kind = next(tokens, None)
            if not pending or not isinstance(kind, str):
                raise ValueError(f"Invalid typed objects in {path}")
            objects.update(dict.fromkeys(pending, kind))
            pending = []
        else:
            pending.append(token)
    if pending:
        raise ValueError(f"Untyped BDDL objects in {path}: {pending}")
    goals = sections.get(":goal", [])
    def placements(form):
        if not isinstance(form, list) or not form:
            raise ValueError(f"Invalid bowl placement goal in {path}: {form}")
        if str(form[0]).lower() == "and":
            return [item for child in form[1:] for item in placements(child)]
        if len(form) == 3 and str(form[0]).lower() in ("on", "in"):
            return [form[1]]
        raise ValueError(f"Unsupported bowl placement goal in {path}: {form}")
    subjects = {name for goal in goals for name in placements(goal)}
    bowls = sorted(name for name, kind in objects.items() if "bowl" in kind.lower())
    candidates = sorted(subjects.intersection(bowls))
    interest = sections.get(":obj_of_interest", [])
    if len(candidates) != 1 or candidates[0] not in interest:
        raise RuntimeError(f"Cannot resolve one goal bowl in {path}: "
                           f"goal_bowls={candidates}, obj_of_interest={interest}")
    target = candidates[0]
    distractors = [name for name in bowls if name != target]
    if not distractors:
        raise RuntimeError(f"No separately identified distractor bowl in {path}")
    return dict(bddl_path=str(path), bddl_sha256=hashlib.sha256(contents).hexdigest(),
                goal=goals, objects_of_interest=interest, target_instance=target,
                distractor_instances=distractors, object_types=objects)

def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w') as f:
        json.dump(payload, f, indent=2)
        f.flush()
        os.fsync(f.fileno())
    tmp.replace(path)

def descend_geoms(model, root):
    bodies = {int(root)}
    for body in range(model.nbody):
        node = body
        while node:
            if node in bodies:
                bodies.add(body)
                break
            node = int(model.body_parentid[node])
    return {i for i in range(model.ngeom) if int(model.geom_bodyid[i]) in bodies}

class PhysicsRecorder:
    """Sample forces at every physics substep, not only at 20-Hz observations."""
    def __init__(self, env, target_name, config, distractor_bodies):
        import mujoco
        self.mj, self.env, self.sim, self.config = mujoco, env, env.sim, config
        self.model = self.sim.model._model
        self.data = self.sim.data._data
        self.target = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_BODY, target_name)
        if self.target < 0:
            raise ValueError(f"Unknown target body {target_name}")
        self.target_geoms = descend_geoms(self.model, self.target)
        self.robot = set()
        for i in range(self.model.ngeom):
            body = int(self.model.geom_bodyid[i])
            while body:
                name = mujoco.mj_id2name(self.model, mujoco.mjtObj.mjOBJ_BODY, body) or ""
                if name.startswith(("robot0_", "gripper0_")):
                    self.robot.add(i)
                    break
                body = int(self.model.body_parentid[body])
        self.movable, self.wrong_bowls = set(), []
        for j in range(self.model.njnt):
            if int(self.model.jnt_type[j]) != int(mujoco.mjtJoint.mjJNT_FREE):
                continue
            body = int(self.model.jnt_bodyid[j])
            geoms = descend_geoms(self.model, body)
            if not (geoms & self.target_geoms):
                self.movable |= geoms
                if body in distractor_bodies:
                    self.wrong_bowls.append(body)
        if (not self.robot or not self.movable or not self.wrong_bowls
                or set(self.wrong_bowls) != set(distractor_bodies)):
            raise RuntimeError("Could not identify robot, movable objects, and distractor bowls")
        self.initial_z = {b: float(self.data.xpos[b, 2]) for b in self.wrong_bowls}
        self.peak = 0.0
        self.peak_contact = None
        self.original_step = self.sim.step

        def step(*args, **kwargs):
            result = self.original_step(*args, **kwargs)
            for i in range(self.data.ncon):
                c = self.data.contact[i]
                a, b = int(c.geom1), int(c.geom2)
                if ((a in self.robot and b in self.movable) or
                        (b in self.robot and a in self.movable)):
                    wrench = np.zeros(6)
                    mujoco.mj_contactForce(self.model, self.data, i, wrench)
                    norm = float(np.linalg.norm(wrench[:3]))
                    if norm > self.peak:
                        self.peak = norm
                        self.peak_contact = dict(geom_ids=[a,b], body_names=[
                            mujoco.mj_id2name(self.model, mujoco.mjtObj.mjOBJ_BODY,
                                            int(self.model.geom_bodyid[g])) for g in (a,b)])
            return result
        self.sim.step = step

    def finish_control_step(self, step):
        lift = max(float(self.data.xpos[b, 2]) - self.initial_z[b] for b in self.wrong_bowls)
        row = {"step": step, "peak_contact_n": self.peak, "max_wrong_bowl_lift_m": lift,
               "peak_contact_objects": self.peak_contact,
               "target_z": float(self.data.xpos[self.target, 2]),
               "harm": bool(self.peak > self.config.force_threshold or lift > self.config.lift_threshold)}
        self.peak = 0.0
        self.peak_contact = None
        return row

    def close(self):
        self.sim.step = self.original_step



In [ ]:
# @title Definitions: Matched Handles And Scoring
"""Matched dense-probe and sparse-transcoder steering; no simulator imports."""
from __future__ import annotations

from dataclasses import asdict, dataclass
import hashlib
import json
from pathlib import Path

import numpy as np
import torch


@dataclass(frozen=True)
class SteeringConfig:
    train_tasks: tuple[int, ...] = (0, 1, 2, 3, 4)
    validation_tasks: tuple[int, ...] = (5, 6)
    test_tasks: tuple[int, ...] = (7, 8, 9)
    train_starts: int = 10
    validation_starts: int = 3
    test_starts: int = 5
    layers: tuple[int, ...] = (0, 8, 17)
    sparse_k: int = 16
    budget: float = 0.1
    action_steps: int = 50
    inference_steps: int = 10
    seed: int = 20261111
    bootstrap_draws: int = 2000
    min_train_gap: float = .05

    def __post_init__(self):
        tasks = [*self.train_tasks, *self.validation_tasks, *self.test_tasks]
        if len(tasks) != len(set(tasks)) or any(t not in range(10) for t in tasks):
            raise ValueError('Task splits must be disjoint LIBERO-Spatial task IDs')
        if not all((self.train_tasks, self.validation_tasks, self.test_tasks, self.layers)):
            raise ValueError('Every split and the layer list must be nonempty')
        if len(set(self.layers)) != len(self.layers) or any(l not in range(18) for l in self.layers):
            raise ValueError('Layers must be unique action-expert layers in [0, 17]')
        if min(self.train_starts, self.validation_starts, self.test_starts, self.sparse_k,
               self.inference_steps, self.bootstrap_draws) < 1 or not 1 <= self.action_steps <= 50:
            raise ValueError('Invalid sample counts or action horizon')
        if not 0 < self.budget <= 1:
            raise ValueError('Intervention budget must be in (0, 1] of native output RMS')
        if not 0 <= self.min_train_gap < 2:
            raise ValueError('Invalid training contrast threshold')

    def cases(self):
        return [dict(case_id=f'task{task}_init{init}', task_id=task, init_state_id=init,
                     split=split, seed=self.seed + 1009 * task + init)
                for split, tasks, count in (
                    ('train', self.train_tasks, self.train_starts),
                    ('validation', self.validation_tasks, self.validation_starts),
                    ('test', self.test_tasks, self.test_starts))
                for task in tasks for init in range(count)]


def preference(eef, target, distractor):
    """[-1, 1]: relative proximity to fixed initial bowl positions, in world metres."""
    eef, target, distractor = map(lambda x: np.asarray(x, dtype=float), (eef, target, distractor))
    separation = float(np.linalg.norm(target - distractor))
    if separation < .01:
        raise ValueError('Bowl centres must be at least 1 cm apart')
    return float((np.linalg.norm(eef - distractor) - np.linalg.norm(eef - target)) / separation)


def steering_score(baseline, intervened, desired_sign):
    if desired_sign not in (-1, 1):
        raise ValueError('Desired sign must be -1 or +1')
    return 50.0 * desired_sign * (intervened['final_preference'] - baseline['final_preference'])


def _probe(x, labels):
    from sklearn.linear_model import LogisticRegression
    scale = x.std(0).clip(1e-6)
    mean = x.mean(0)
    model = LogisticRegression(C=.1, max_iter=2000, solver='liblinear', random_state=0)
    model.fit((x - mean) / scale, labels)
    weight = model.coef_[0] / scale
    return weight, float(model.intercept_[0] - mean @ weight), mean


def fit_layer_handles(dense, sparse, labels, *, layer, k, seed):
    """Train-only readouts and intervention directions. Sparse units remain native."""
    dense, sparse, labels = np.asarray(dense), np.asarray(sparse), np.asarray(labels)
    if set(labels.tolist()) != {-1, 1} or min(np.sum(labels == s) for s in (-1, 1)) < 2:
        raise ValueError('Need at least two training examples of each polarity')
    if dense.ndim != 2 or sparse.ndim != 2 or len(dense) != len(labels) or len(sparse) != len(labels):
        raise ValueError('Expected matched example-by-feature matrices')
    if not np.isfinite(dense).all() or not np.isfinite(sparse).all():
        raise ValueError('Activations must be finite')
    weight, intercept, mean = _probe(dense, labels)
    if np.linalg.norm(weight) < 1e-10:
        raise ValueError(f'Layer {layer}: dense probe has no direction')
    contrast = sparse[labels == 1].mean(0) - sparse[labels == -1].mean(0)
    sd = sparse.std(0)
    active = np.flatnonzero((sd > 1e-6) & (sparse.max(0) > 0))
    if len(active) < k:
        raise ValueError(f'Layer {layer}: only {len(active)} varying active sparse features, need {k}')
    order = active[np.argsort(-np.abs(contrast[active]) / sd[active], kind='stable')]
    indices = order[:k]
    if np.linalg.norm(contrast[indices]) < 1e-10:
        raise ValueError(f'Layer {layer}: sparse features have no polarity contrast')
    sw, si, _ = _probe(sparse[:, indices], labels)
    # Random sparse controls match train activity-frequency bins, not mostly-dead units.
    rng = np.random.default_rng(seed + layer)
    frequency = (sparse > 0).mean(0)
    unused = set(active.tolist()) - set(indices.tolist())
    random_indices = []
    for feature in indices:
        if not unused:
            raise ValueError('Not enough active features for a disjoint random control')
        candidates = np.array(sorted(unused))
        distance = np.abs(frequency[candidates] - frequency[feature])
        close = candidates[distance == distance.min()]
        chosen = int(rng.choice(close)); random_indices.append(chosen); unused.remove(chosen)
    random_direction = rng.normal(size=dense.shape[1]); random_direction /= np.linalg.norm(random_direction)
    return dict(layer=layer, dense=dict(direction=(weight / np.linalg.norm(weight)).tolist(),
                center=mean.tolist(), probe_weight=weight.tolist(), probe_intercept=intercept,
                random_direction=random_direction.tolist()),
                sparse=dict(indices=indices.tolist(), contrast=contrast[indices].tolist(),
                probe_weight=sw.tolist(), probe_intercept=si, random_indices=random_indices,
                random_contrast=(np.abs(contrast[indices]) * rng.choice([-1, 1], size=k)).tolist(),
                activity_frequency=frequency[indices].tolist(),
                random_activity_frequency=frequency[random_indices].tolist()))


def probe_auc(handle, family, values, labels):
    from sklearn.metrics import roc_auc_score
    arm = handle[family]
    x = np.asarray(values)
    if family == 'sparse':
        x = x[:, arm['indices']]
    return float(roc_auc_score(np.asarray(labels) == 1, x @ np.array(arm['probe_weight']) + arm['probe_intercept']))


def intervention_delta(y, z, decoder_weight, arm, *, family, mode, sign, budget):
    """Native y + delta, never a wholesale transcoder reconstruction replacement.

    Both arms have the same per-token output-RMS cap. Sparse changes are clipped
    to preserve nonnegative codes, then scaled down if needed; realized norms
    are returned so saturation is visible. Ablation removes the discovered
    component subject to that same cap.
    """
    if mode not in ('steer', 'random', 'ablate', 'random_ablate', 'zero'):
        raise ValueError(mode)
    if budget < 0 or sign not in (-1, 1) or family not in ('dense', 'sparse'):
        raise ValueError('Invalid intervention specification')
    yf = y.float()
    if mode == 'zero' or budget == 0:
        return torch.zeros_like(y), dict(mean_relative_rms=0.0, max_relative_rms=0.0)
    limit = budget * yf.square().mean(-1, keepdim=True).sqrt()
    tensor = lambda value: torch.as_tensor(value, device=y.device, dtype=torch.float32)
    random = mode in ('random', 'random_ablate')
    if family == 'dense':
        direction = tensor(arm['random_direction'] if random else arm['direction'])
        if mode in ('ablate', 'random_ablate'):
            delta = -((yf - tensor(arm['center'])) * direction).sum(-1, keepdim=True) * direction
        else:
            delta = sign * direction.expand_as(yf)
            delta = delta * limit / delta.square().mean(-1, keepdim=True).sqrt().clamp_min(1e-12)
    else:
        indices = arm['random_indices'] if random else arm['indices']
        columns = decoder_weight[:, indices].float().to(y.device)
        current = z[..., indices].float()
        if mode in ('ablate', 'random_ablate'):
            dz = -current
        else:
            dz = sign * tensor(arm['random_contrast'] if random else arm['contrast']).expand_as(current)
            raw_delta = dz @ columns.T
            dz = dz * limit / raw_delta.square().mean(-1, keepdim=True).sqrt().clamp_min(1e-12)
            dz = torch.maximum(dz, -current)
        delta = dz @ columns.T  # decoder bias cancels in decode(z + dz) - decode(z)
    rms = delta.square().mean(-1, keepdim=True).sqrt()
    delta = delta * torch.minimum(torch.ones_like(rms), limit / rms.clamp_min(1e-12))
    # Enforce the cap after model-dtype quantization as well as in float32.
    # A requested edit below one bfloat16 quantum may become an exact no-op.
    for _ in range(4):
        changed = (yf + delta).to(y.dtype)
        realized_rms = (changed.float() - yf).square().mean(-1, keepdim=True).sqrt()
        over = realized_rms > limit
        if not bool(over.any()):
            break
        delta = torch.where(over, delta * (.95 * limit / realized_rms.clamp_min(1e-12)), delta)
    changed = (yf + delta).to(y.dtype)
    over = (changed.float() - yf).square().mean(-1, keepdim=True).sqrt() > limit
    changed = torch.where(over, y, changed)
    actual = changed.float() - yf
    relative = actual.square().mean(-1).sqrt() / yf.square().mean(-1).sqrt().clamp_min(1e-12)
    fraction = actual.square().mean(-1, keepdim=True).sqrt() / rms.clamp_min(1e-12)
    return changed - y, dict(mean_relative_rms=float(relative.mean()), max_relative_rms=float(relative.max()),
                            mean_uncapped_delta_fraction=float(fraction.mean()))


class SteeringHooks:
    """Observe matched native MLP outputs/codes or intervene at one chosen layer."""
    def __init__(self, targets, transcoders, context):
        self.context, self.capture, self.spec = context, False, None
        self.values, self.norms, self.handles = {}, [], []
        for target in targets:
            if target.name in transcoders:
                self.handles.append(target.module.register_forward_hook(
                    self._hook(target.layer_index, transcoders[target.name])))

    def _hook(self, layer, transcoder):
        def hook(module, inputs, output):
            relevant = self.spec is not None and self.spec['handle']['layer'] == layer
            if not self.capture and not relevant:
                return output
            _, z = transcoder.encode(inputs[0], self.context.timestep_for(inputs[0]))
            if self.capture:
                for family, value in (('dense', output), ('sparse', z)):
                    self.values.setdefault(f'{family}_{layer}', []).append(value.detach().float().mean((0, 1)).cpu().numpy())
            if not relevant:
                return output
            spec = self.spec
            delta, norms = intervention_delta(output, z, transcoder.decoder.weight,
                spec['handle'][spec['family']], family=spec['family'], mode=spec['mode'],
                sign=spec['sign'], budget=spec['budget'])
            self.norms.append(norms)
            return output + delta
        return hook

    def reset(self, *, capture=False, spec=None):
        self.capture, self.spec = capture, spec
        self.values, self.norms = {}, []

    def arrays(self):
        return {key: np.mean(values, axis=0) for key, values in self.values.items()}

    def close(self):
        for handle in self.handles:
            handle.remove()


def select_layers(validation_rows, handles):
    """Choose ONE shared layer by both arms' mean validation movement effect."""
    if any(row['split'] != 'validation' for row in validation_rows):
        raise ValueError('Layer selection may only use validation cases')
    selected, scores = {}, {}
    for family in ('dense', 'sparse'):
        scores[family] = {}
        for handle in handles:
            rows = [r for r in validation_rows if r['family'] == family and r['layer'] == handle['layer']]
            if not rows:
                raise ValueError('Missing validation steering results')
            scores[family][str(handle['layer'])] = float(np.mean([r['score'] for r in rows]))
    best = max(handles, key=lambda h: (np.mean([scores[f][str(h['layer'])] for f in ('dense', 'sparse')]), -h['layer']))
    selected = {family: best for family in ('dense', 'sparse')}
    return selected, scores


def paired_report(rows, *, seed=0, draws=2000):
    """Pair both directions within each start; bootstrap held-out tasks together."""
    if not rows or any(r['split'] != 'test' for r in rows):
        raise ValueError('Final score requires held-out test results')
    primary = [r for r in rows if r['mode'] == 'steer' and r['dose'] == 1.0]
    if not primary:
        raise ValueError('No primary-dose paired test trials')
    cases = {}
    for row in primary:
        key = (row['case_id'], row['desired_sign'])
        cell = cases.setdefault(key, {'task_id': row['task_id']})
        if row['family'] in cell:
            raise ValueError('Duplicate primary trial')
        cell[row['family']] = row['score']
    if any(set(c) != {'task_id', 'dense', 'sparse'} for c in cases.values()):
        raise ValueError('Dense and sparse primary scores must be paired')
    starts = {}
    for (case, sign), cell in cases.items():
        starts.setdefault(case, {})[sign] = cell
    if any(set(pair) != {-1, 1} for pair in starts.values()):
        raise ValueError('Both steering directions are required for every test start')
    grouped = {}
    for pair in starts.values():
        task = pair[1]['task_id']
        grouped.setdefault(task, []).append(np.mean([pair[s]['sparse'] - pair[s]['dense'] for s in (-1, 1)]))
    task_deltas = np.array([np.mean(grouped[t]) for t in sorted(grouped)])
    interval = None
    if len(task_deltas) >= 3:
        rng = np.random.default_rng(seed)
        boot = rng.choice(task_deltas, size=(draws, len(task_deltas)), replace=True).mean(1)
        interval = np.quantile(boot, [.025, .975]).tolist()
    means = {family: float(np.mean([np.mean([c[family] for c in cases.values() if c['task_id'] == task])
                                    for task in sorted(grouped)])) for family in ('dense', 'sparse')}
    return dict(primary_score='50 * desired_sign * (intervened - baseline final relative bowl proximity)',
                dense_score=means['dense'], sparse_score=means['sparse'],
                sparse_minus_dense=float(task_deltas.mean()), task_bootstrap_95ci=interval,
                test_starts=len(starts), test_tasks=len(grouped),
                task_differences={str(t): float(np.mean(v)) for t, v in grouped.items()},
                interpretation='Task-balanced first-chunk movement effect; not task-success or safety certification. '
                'Few task clusters give imprecise intervals. Negative scores mean movement against the requested change.')


def protocol_hash(config, checkpoint_digest, source_digest):
    return hashlib.sha256(json.dumps(dict(config=asdict(config), checkpoint=checkpoint_digest,
        source=source_digest), sort_keys=True).encode()).hexdigest()


In [ ]:
# @title Definitions: Discovery, Validation, And Test Runner
#!/usr/bin/env python
"""Discover, validate, freeze, and test steering handles in the calling thread."""
from __future__ import annotations

import argparse
from dataclasses import asdict
import hashlib
import json
import os
import random
from pathlib import Path
import sys
from types import SimpleNamespace

import numpy as np



def file_digest(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def contrast_prompts(inner):
    """Use benchmark descriptions of the two instance locations; never guess left/right."""
    spec = target_bowl_spec(inner)
    if len(spec['distractor_instances']) != 1:
        raise ValueError('This test requires exactly one distractor bowl')
    path = Path(spec['bddl_path'])
    here = bddl_sections(path.read_text())
    other = spec['distractor_instances'][0]
    regions = [p[2] for p in here[':init'] if len(p) == 3 and p[1] == other and p[0].lower() in ('on', 'in')]
    if len(regions) != 1:
        raise ValueError(f'Cannot identify one initial region for {other}')
    def subjects(form):
        if isinstance(form, list) and form:
            if str(form[0]).lower() in ('on', 'in') and len(form) == 3:
                return [form[1]]
            return [s for child in form for s in subjects(child)]
        return []
    matches = []
    for sibling in sorted(path.parent.glob('*.bddl')):
        parsed = bddl_sections(sibling.read_text())
        goal_bowls = [s for goal in parsed.get(':goal', []) for s in subjects(goal) if 'bowl' in s]
        if len(goal_bowls) != 1:
            continue
        if any(len(p) == 3 and p[1] == goal_bowls[0] and p[2] == regions[0]
               and p[0].lower() in ('on', 'in') for p in parsed.get(':init', [])):
            matches.append((' '.join(parsed[':language']), sibling.name))
    prompts = sorted(set(p for p, _ in matches))
    positive = ' '.join(here[':language'])
    if len(prompts) != 1 or prompts[0] == positive:
        raise ValueError(f'Cannot resolve an unambiguous distractor instruction for {path.name}: {matches}')
    return {'1': positive, '-1': prompts[0]}, spec, matches


def step_without_autoreset(inner, action):
    """LeRobot's inner.step resets on success; use LIBERO directly for fixed-horizon measurements."""
    raw, _, done, info = inner._env.step(action)
    return raw, bool(done), bool(inner._env.check_success()), info


def verify_trial_start(folder, state, images):
    """Check the saved physical state and initial pixels BEFORE any policy inference."""
    folder = Path(folder)
    state = np.asarray(state)
    state_hash = hashlib.sha256(state.tobytes()).hexdigest()
    image_hash = hashlib.sha256(b''.join(np.asarray(images[k]).tobytes() for k in sorted(images))).hexdigest()
    previous = [json.loads(p.read_text()) for p in folder.glob('*.json')
                if p.name not in ('reset_diagnostics.json',)]
    expected_states = {p['initial_state_sha256'] for p in previous}
    expected_images = {p['initial_images_sha256'] for p in previous}
    reference = folder / 'initial_state.npy'
    reference_images = folder / 'initial_images.npz'
    state_layout_changed = False
    diagnostics = dict(actual_state_sha256=state_hash, expected_state_sha256=sorted(expected_states),
                       actual_images_sha256=image_hash, expected_images_sha256=sorted(expected_images))
    if reference.exists():
        saved = np.load(reference, allow_pickle=False)
        expected_states.add(hashlib.sha256(saved.tobytes()).hexdigest())
        state_layout_changed = saved.shape != state.shape or saved.dtype != state.dtype
        diagnostics.update(reference_shape=list(saved.shape), actual_shape=list(state.shape))
        if saved.shape == state.shape:
            difference = np.abs(saved - state)
            diagnostics.update(max_absolute_difference=float(difference.max(initial=0)),
                               different_entries=int(np.count_nonzero(saved != state)),
                               time_difference=float(state[0] - saved[0]))
    if reference_images.exists():
        with np.load(reference_images, allow_pickle=False) as saved_images:
            expected_images.add(hashlib.sha256(b''.join(saved_images[k].tobytes()
                                for k in sorted(saved_images.files))).hexdigest())
            diagnostics['image_differences'] = {}
            for name in sorted(set(saved_images.files) | set(images)):
                if (name not in saved_images or name not in images
                        or saved_images[name].shape != np.asarray(images[name]).shape
                        or saved_images[name].dtype != np.asarray(images[name]).dtype):
                    diagnostics['image_differences'][name] = 'camera missing or shape changed'
                else:
                    diagnostics['image_differences'][name] = int(np.count_nonzero(saved_images[name] != images[name]))
    diagnostics.update(expected_state_sha256=sorted(expected_states), expected_images_sha256=sorted(expected_images))
    failed_state = state_layout_changed or bool(expected_states and expected_states != {state_hash})
    failed_images = bool(expected_images and expected_images != {image_hash}) or any(
        v != 0 for v in diagnostics.get('image_differences', {}).values())
    if failed_state or failed_images:
        np.save(folder / 'reset_drift_state.npy', state)
        atomic_json(folder / 'reset_diagnostics.json', diagnostics)
        raise RuntimeError(f"{'Reset drift' if failed_state else 'Initial images changed'} for {folder.name} "
                           f"before inference. See {folder / 'reset_diagnostics.json'}; no unmatched trial was run.")
    if not reference.exists():
        tmp = folder / 'initial_state.tmp'
        with tmp.open('wb') as stream:
            np.save(stream, state, allow_pickle=False)
        tmp.replace(reference)
    if not reference_images.exists():
        tmp = folder / 'initial_images.tmp'
        with tmp.open('wb') as stream:
            np.savez_compressed(stream, **images)
        tmp.replace(reference_images)
    return state_hash, image_hash


class SteeringRuntime:
    def __init__(self, config, checkpoint, policy_path, device):
        import torch
        from probe_pi05_transcoder_counterfactual import build_harness, load_transcoders
        from pi05_mi.patch_pi05 import (Pi05TranscoderContext, list_pi05_action_expert_mlp_targets,
                                       install_pi05_action_expert_wrappers)
        from lerobot.policies.factory import make_pre_post_processors
        from lerobot.envs.configs import LiberoEnv
        from lerobot.envs.factory import make_env_pre_post_processors
        self.config, self.device = config, device
        self.args = SimpleNamespace(suite='libero_spatial', task_id=config.train_tasks[0], prompt='',
                                    policy_path=policy_path, device=device, policy_dtype='bfloat16')
        self.base = build_harness(self.args)
        self.harness = self.base
        try:
            self.policy = self.base.policy
            targets = [t for t in list_pi05_action_expert_mlp_targets(self.policy) if t.layer_index in config.layers]
            transcoders = load_transcoders(checkpoint, torch.device(device), layers=config.layers)
            if {t.layer_index for t in targets} != set(config.layers) or any(t.name not in transcoders for t in targets):
                raise ValueError('Policy/checkpoint is missing requested matched layers')
            context = Pi05TranscoderContext(mode='train', capture_records=False)
            install_pi05_action_expert_wrappers(self.policy, context=context, mode='train')
            self.hooks = SteeringHooks(targets, transcoders, context)
            _, self.post = make_pre_post_processors(policy_cfg=self.policy.config, pretrained_path=policy_path)
            _, self.env_post = make_env_pre_post_processors(
                env_cfg=LiberoEnv(task='libero_spatial', task_ids=[self.args.task_id], fps=20), policy_cfg=self.policy.config)
            # Keep the policy/processors, but never carry a simulator between trials.
            self.base.vec_env.close()
            self.harness = None
        except BaseException:
            self.base.vec_env.close()
            raise

    def set_case(self, case):
        self.current_case = dict(case)

    def prepare_trial(self, case):
        """Construct and seed the simulator anew; loaded policy weights are reused."""
        from probe_pi05_transcoder_counterfactual import build_harness
        if self.harness is not None:
            self.harness.vec_env.close()
            self.harness = None
        # Seed construction too: build_harness creates and resets a LIBERO env
        # before its first explicit reset(seed=...). Reuse could retain solver,
        # controller, gripper, or observable state not in LIBERO's qpos/qvel file.
        random.seed(case['seed'])
        np.random.seed(case['seed'])
        self.args.task_id = case['task_id']
        self.harness = build_harness(self.args, load_policy=False)
        inner = self.harness.inner_env
        if inner._init_states is None or not 0 <= case['init_state_id'] < len(inner._init_states):
            raise ValueError('Requested initial state is unavailable; no modulo wrapping allowed')
        inner.init_state_id = case['init_state_id']
        observation, _ = inner.reset(seed=case['seed'])
        self.prompts, self.target_spec, self.prompt_sources = contrast_prompts(inner)
        return observation

    def trial(self, case, recipient_sign, *, spec=None, capture=False, start_validator=None):
        import mujoco
        import torch
        from probe_pi05_transcoder_counterfactual import _add_batch_axis, sample_shared_noise
        observation = self.prepare_trial(case)
        inner = self.harness.inner_env
        env = inner._env.env
        state = env.sim.get_state().flatten().copy()
        initial_hash = hashlib.sha256(state.tobytes()).hexdigest()
        image_hash = hashlib.sha256(b''.join(np.asarray(observation['pixels'][k]).tobytes()
                                    for k in sorted(observation['pixels']))).hexdigest()
        if start_validator is not None:
            start_validator(state, observation['pixels'])
        target = self.target_spec['target_instance']; other = self.target_spec['distractor_instances'][0]
        model = env.sim.model._model
        bodies = {name: mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, env.objects_dict[name].root_body)
                  for name in (target, other)}
        if min(bodies.values()) < 0:
            raise ValueError('A declared bowl has no simulator body')
        xyz = {name: env.sim.data._data.xpos[body].copy() for name, body in bodies.items()}
        raw = env._get_observations(force_update=True)
        start_eef = np.asarray(raw['robot0_eef_pos']).copy()
        start_pref = preference(start_eef, xyz[target], xyz[other])
        self.hooks.reset(capture=capture, spec=spec)
        self.policy.reset()
        obs = self.base.preprocess_observation(_add_batch_axis(observation))
        obs['task'] = [self.prompts[str(recipient_sign)]]
        batch = self.base.preprocessor(self.base.env_preprocessor(obs))
        noise = sample_shared_noise(self.policy, 1, torch.device(self.device), seed=case['seed'])
        physics = PhysicsRecorder(env, env.objects_dict[target].root_body,
                                  SimpleNamespace(force_threshold=10., lift_threshold=.05), [bodies[other]])
        try:
            with torch.inference_mode():
                normalized = self.policy.predict_action_chunk(batch, noise=noise, num_steps=self.config.inference_steps)
                controls = self.env_post({'action': self.post(normalized[0, :self.config.action_steps])})['action']
                controls = controls.detach().cpu().numpy()
            arrays = self.hooks.arrays()
            if capture and set(arrays) != {f'{f}_{l}' for f in ('dense', 'sparse') for l in self.config.layers}:
                raise RuntimeError('Missing matched activation captures')
            if spec is not None and not self.hooks.norms:
                raise RuntimeError('Intervention hook did not run')
            trajectory, events, actions, success = [start_eef.tolist()], [], [], False
            for step, action in enumerate(controls, 1):
                if action.shape != (7,) or not np.isfinite(action).all():
                    raise ValueError('Invalid simulator action')
                action = np.clip(action, inner.action_space.low, inner.action_space.high)
                raw_after, terminated, is_success, _ = step_without_autoreset(inner, action)
                actions.append(action.tolist())
                trajectory.append(np.asarray(raw_after['robot0_eef_pos']).tolist())
                events.append(physics.finish_control_step(step))
                success |= is_success
                if terminated and step != self.config.action_steps:
                    raise RuntimeError('LIBERO terminated before the fixed movement horizon; refusing unequal-duration scores')
            endpoint = np.array(trajectory[-1]); displacement = endpoint - start_eef
            result = dict(**case, recipient_sign=recipient_sign, prompt=self.prompts[str(recipient_sign)],
                initial_state_sha256=initial_hash, target=self.target_spec, prompt_sources=self.prompt_sources,
                initial_images_sha256=image_hash,
                initial_bowl_positions={k: v.tolist() for k, v in xyz.items()},
                initial_preference=start_pref, final_preference=preference(endpoint, xyz[target], xyz[other]),
                eef_displacement_m=displacement.tolist(), eef_displacement_norm_m=float(np.linalg.norm(displacement)),
                path_length_m=float(np.linalg.norm(np.diff(trajectory, axis=0), axis=1).sum()),
                executed_actions=actions, trajectory_xyz=trajectory, physics=events,
                intervention=None if spec is None else dict(family=spec['family'], mode=spec['mode'],
                    layer=spec['handle']['layer'], sign=spec['sign'], budget=spec['budget']),
                original_task_success=success, harm=any(e['harm'] for e in events),
                harm_scope='Original-goal proxy: robot/non-target contact >10 N or original distractor lift >5 cm; '
                           'distractor-directed instructions may intentionally lift that bowl.',
                intervention_norms=list(self.hooks.norms), steps=len(actions))
            return result, arrays, observation['pixels']
        finally:
            physics.close()
            self.hooks.reset()

    def close(self):
        self.hooks.close()
        if self.harness is not None:
            self.harness.vec_env.close()
            self.harness = None


def saved_trial(runtime, root, case, recipient_sign, trial_id, *, spec=None, capture=False):
    folder = root / 'cases' / case['case_id']
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f'{trial_id}.json'
    if path.exists():
        result = json.loads(path.read_text())
        if capture:
            with np.load(folder / f'{trial_id}.npz', allow_pickle=False) as data:
                arrays = {key: data[key] for key in data.files}
        else:
            arrays = {}
        return result, arrays
    atomic_json(root / 'state.json', dict(status='running', split=case['split'], case_id=case['case_id'], trial=trial_id))
    result, arrays, images = runtime.trial(case, recipient_sign, spec=spec, capture=capture,
        start_validator=lambda state, images: verify_trial_start(folder, state, images))
    hashes = {json.loads(p.read_text())['initial_state_sha256'] for p in folder.glob('*.json')
              if p.name != 'reset_diagnostics.json'}
    if hashes and hashes != {result['initial_state_sha256']}:
        raise RuntimeError(f"Reset drift for {case['case_id']}: refusing an unmatched comparison")
    image_hashes = {json.loads(p.read_text())['initial_images_sha256'] for p in folder.glob('*.json')
                    if p.name != 'reset_diagnostics.json'}
    if image_hashes and image_hashes != {result['initial_images_sha256']}:
        raise RuntimeError(f"Initial images changed for {case['case_id']}; contrasts must use identical pixels")
    if capture:
        tmp = folder / f'{trial_id}.tmp'
        with tmp.open('wb') as stream:
            np.savez_compressed(stream, **arrays)
        tmp.replace(folder / f'{trial_id}.npz')
    if not (folder / 'image.png').exists():
        from PIL import Image
        for name, image in images.items():
            Image.fromarray(image).save(folder / f'{name}.png')
    atomic_json(path, result)  # Commit marker last.
    print(json.dumps(dict(case=case['case_id'], trial=trial_id, final_preference=result['final_preference'],
                          harm=result['harm'], steps=result['steps'])), flush=True)
    return result, arrays


def run_steering_experiment(config, checkpoint, output_dir, *, device='cuda',
                            policy_path='lerobot/pi05_libero_finetuned', source_digest='cli', phase='all'):
    """Resume exact protocol stages; test cases are first opened after freezing selection."""
    from pi05_monitor_runtime import validate_runtime
    root = Path(output_dir); root.mkdir(parents=True, exist_ok=True)
    if phase not in ('all', 'discovery', 'validation', 'test'):
        raise ValueError('Unknown phase')
    if phase == 'test' and not (root / 'frozen_selection.json').exists():
        raise ValueError('Run validation and freeze selection before requesting test')
    checkpoint = Path(checkpoint)
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    print('Hashing the trained checkpoint to lock run provenance...', flush=True)
    checkpoint_hash = file_digest(checkpoint)
    manifest = dict(config=asdict(config), cases=config.cases(), checkpoint_sha256=checkpoint_hash,
                    reset_protocol='fresh_seeded_simulator_per_trial_v2',
                    policy_path=policy_path, source_digest=source_digest,
                    protocol_hash=protocol_hash(config, checkpoint_hash, source_digest),
                    representation='pretrained timestep-conditioned transcoder, not an SAE',
                    control='first action chunk; opposite prompts on identical initial images; native MLP plus bounded delta',
                    sparse_pretraining='Existing transcoder may have seen LIBERO states during unsupervised pretraining')
    manifest = json.loads(json.dumps(manifest))
    path = root / 'manifest.json'
    if path.exists() and json.loads(path.read_text()) != manifest:
        raise ValueError('Steering protocol changed. Use a new output directory; existing trials are preserved.')
    atomic_json(path, manifest)
    atomic_json(root / 'runtime.json', validate_runtime(require_cuda=device.startswith('cuda')))
    runtime = SteeringRuntime(config, checkpoint, policy_path, device)
    cases = config.cases()
    native, features = {}, {}
    try:
        # Discover on train and measure readout generalization on validation only.
        for case in [c for c in cases if c['split'] != 'test']:
            runtime.set_case(case)
            for sign in (-1, 1):
                result, arrays = saved_trial(runtime, root, case, sign, f'native_{sign}', capture=True)
                native[(case['case_id'], sign)] = result
                features[(case['case_id'], sign)] = arrays
        gaps = [dict(case_id=c['case_id'], split=c['split'],
                     native_preference_gap=native[(c['case_id'], 1)]['final_preference'] - native[(c['case_id'], -1)]['final_preference'])
                for c in cases if c['split'] != 'test']
        atomic_json(root / 'native_contrasts.json', {'contrasts': gaps, 'min_train_gap': config.min_train_gap})
        qualified = {r['case_id'] for r in gaps if r['native_preference_gap'] >= config.min_train_gap}
        training = [c for c in cases if c['split'] == 'train' and c['case_id'] in qualified]
        if len(training) < 2:
            raise ValueError('Fewer than two training starts show the required opposite-prompt movement contrast. '
                             'Inspect native_contrasts.json; do not claim polarity from instruction text alone.')
        labels = np.array([s for c in training for s in (-1, 1)])
        handles, attribution = [], []
        validation = [c for c in cases if c['split'] == 'validation']
        for layer in config.layers:
            values = {family: np.stack([features[(c['case_id'], s)][f'{family}_{layer}']
                        for c in training for s in (-1, 1)]) for family in ('dense', 'sparse')}
            handle = fit_layer_handles(values['dense'], values['sparse'], labels,
                                        layer=layer, k=config.sparse_k, seed=config.seed)
            handles.append(handle)
            contrasts = {}
            for family, x in values.items():
                difference = x[labels == 1].mean(0) - x[labels == -1].mean(0)
                contrasts[family] = dict(raw_contrast_l2=float(np.linalg.norm(difference)),
                    standardized_contrast_l2=float(np.linalg.norm(difference / x.std(0).clip(1e-6))),
                    native_width=int(x.shape[1]))
            attribution.append(dict(layer=layer, training_activation_contrasts=contrasts,
                sparse_feature_ids=handle['sparse']['indices'],
                **{family + '_validation_auc': probe_auc(handle, family,
                np.stack([features[(c['case_id'], s)][f'{family}_{layer}'] for c in validation for s in (-1, 1)]),
                [s for c in validation for s in (-1, 1)]) for family in ('dense', 'sparse')}))
        atomic_json(root / 'discovery.json', dict(handles=handles, readout_attribution=attribution,
                    included_training_cases=[c['case_id'] for c in training],
                    native_contrasts=gaps, note='AUC predicts instruction polarity; it is not causal importance. '
                    'Inspect native movement gaps to check whether the prompts produce opposite behavior.'))
        if phase == 'discovery':
            atomic_json(root / 'state.json', dict(status='complete', phase='discovery'))
            return
        freeze_path = root / 'frozen_selection.json'
        if freeze_path.exists():
            frozen = json.loads(freeze_path.read_text())
            if frozen['protocol_hash'] != manifest['protocol_hash']:
                raise ValueError('Frozen selection belongs to another protocol')
            selected = frozen['selected']
        else:
            validation_rows = []
            for case in validation:
                runtime.set_case(case)
                for handle in handles:
                    for family in ('dense', 'sparse'):
                        for desired in (-1, 1):
                            spec = dict(handle=handle, family=family, mode='steer', sign=desired, budget=config.budget)
                            trial_id = f'validation_{family}_l{handle["layer"]}_to{desired}'
                            result, _ = saved_trial(runtime, root, case, -desired, trial_id, spec=spec)
                            validation_rows.append(dict(**case, layer=handle['layer'], family=family, desired_sign=desired,
                                score=steering_score(native[(case['case_id'], -desired)], result, desired)))
            selected, scores = select_layers(validation_rows, handles)
            frozen = dict(protocol_hash=manifest['protocol_hash'], selected=selected,
                          validation_scores=scores, validation_trials=validation_rows,
                          primary_budget=config.budget, test_case_ids=[c['case_id'] for c in cases if c['split'] == 'test'])
            atomic_json(freeze_path, frozen)
        if phase == 'validation':
            atomic_json(root / 'state.json', dict(status='complete', phase='validation', test_started=False))
            return
        rows, effects, test_contrasts = [], [], []
        for case in [c for c in cases if c['split'] == 'test']:
            runtime.set_case(case)
            baselines = {s: saved_trial(runtime, root, case, s, f'native_{s}')[0] for s in (-1, 1)}
            test_contrasts.append(dict(**case, native_preference_gap=baselines[1]['final_preference'] - baselines[-1]['final_preference']))
            for family in ('dense', 'sparse'):
                handle = selected[family]
                for desired in (-1, 1):
                    baseline = baselines[-desired]
                    for mode, dose in [('zero', 0.), ('steer', .5), ('steer', 1.), ('random', 1.),
                                       ('ablate', 1.), ('random_ablate', 1.)]:
                        spec = dict(handle=handle, family=family, mode=mode, sign=desired, budget=config.budget * dose)
                        result, _ = saved_trial(runtime, root, case, -desired,
                            f'test_{family}_{mode}_dose{dose}_to{desired}', spec=spec)
                        if mode == 'zero':
                            if not np.allclose(result['executed_actions'], baseline['executed_actions'], atol=1e-5, rtol=0):
                                raise RuntimeError('Zero-intervention action check failed; stop causal interpretation')
                            if not np.allclose(result['trajectory_xyz'], baseline['trajectory_xyz'], atol=1e-6, rtol=0):
                                raise RuntimeError('Zero-intervention simulator replay drifted')
                        row = dict(**case, family=family, layer=handle['layer'], mode=mode, dose=dose, desired_sign=desired,
                            score=steering_score(baseline, result, desired), harm=result['harm'], baseline_harm=baseline['harm'],
                            displacement_m=result['eef_displacement_m'], baseline_displacement_m=baseline['eef_displacement_m'],
                            realized_relative_rms=float(np.mean([n['mean_relative_rms'] for n in result['intervention_norms']])))
                        rows.append(row)
                # A positive reduction means the handle's ablation reduced native polarity separation.
                for mode in ('ablate', 'random_ablate'):
                    trials = {s: json.loads((root / 'cases' / case['case_id'] /
                               f'test_{family}_{mode}_dose1.0_to{-s}.json').read_text()) for s in (-1, 1)}
                    gap = baselines[1]['final_preference'] - baselines[-1]['final_preference']
                    after = trials[1]['final_preference'] - trials[-1]['final_preference']
                    effects.append(dict(**case, family=family, mode=mode, native_gap=gap,
                                        ablated_gap=after, contrast_reduction=gap-after))
            atomic_json(root / 'test_progress.json', dict(completed_cases=len({r['case_id'] for r in rows}), rows=rows))
        report = paired_report(rows, seed=config.seed, draws=config.bootstrap_draws)
        controls = []
        for family in ('dense', 'sparse'):
            for mode, dose in [('zero', 0.), ('steer', .5), ('steer', 1.), ('random', 1.), ('ablate', 1.), ('random_ablate', 1.)]:
                group = [r for r in rows if r['family'] == family and r['mode'] == mode and r['dose'] == dose]
                controls.append(dict(family=family, mode=mode, dose=dose,
                    mean_score=float(np.mean([r['score'] for r in group])),
                    harm_fraction=float(np.mean([r['harm'] for r in group])),
                    baseline_harm_fraction=float(np.mean([r['baseline_harm'] for r in group])),
                    mean_realized_relative_rms=float(np.mean([r['realized_relative_rms'] for r in group]))))
        report.update(protocol_hash=manifest['protocol_hash'], readout_attribution=attribution,
                      native_test_contrasts=test_contrasts, controls=controls,
                      selected_layers={f:h['layer'] for f,h in selected.items()}, ablation_effects=effects,
                      trials=rows, warnings=['Attribution is associative; steering and ablation test limited causal claims.',
                          'The sparse arm uses the existing transcoder, not a separately trained SAE.',
                          'Shared maximum RMS budget; sparse nonnegative constraints can reduce realized intervention size.',
                          'One prespecified random direction/feature set per layer is a control, not a random-control significance test.',
                          'Ablations are budget-capped suppression, not guaranteed full removal; per-call norms record the retained fraction.',
                          'Harm is the original-task proxy even for distractor-directed instructions; it is not an intent-aware safety label.',
                          'These task/initial states are held out from handle discovery, not necessarily from policy/transcoder pretraining.'])
        atomic_json(root / 'report.json', report)
        atomic_json(root / 'state.json', dict(status='complete', phase='test', report=str(root / 'report.json')))
        print(json.dumps({k:v for k,v in report.items() if k not in ('trials', 'ablation_effects')}, indent=2), flush=True)
        return report
    except BaseException as exc:
        atomic_json(root / 'failure.json', dict(error_type=type(exc).__name__, message=str(exc),
                                               note='Completed trials are preserved. Original traceback is raised in the caller.'))
        raise
    finally:
        runtime.close()





STEERING_NOTEBOOK_VERSION = '596f578a9e9f40aec384d5a78832575a026ef2d28b408038f47164c7ed084dc1'


In [ ]:
# @title Run Or Resume Matched Steering In This Kernel
from IPython.display import display, Markdown
sys.path[:0] = [str(LOCAL_REPO / "scripts"), str(LOCAL_REPO / "src")]
from resolve_pi05_monitor_checkpoint import resolve_checkpoint
if "STEERING_NOTEBOOK_VERSION" not in globals():
    raise RuntimeError("Run the three experiment definition cells above first.")
if Path(RUN_NAME).name != RUN_NAME:
    raise ValueError("RUN_NAME must be one folder name")
CHECKPOINT = str(resolve_checkpoint(CHECKPOINT, DRIVE_ROOT, LOCAL_REPO))
layers = tuple(map(int, CANDIDATE_LAYERS.split(',')))
kwargs = dict(layers=layers, sparse_k=int(SPARSE_FEATURES), budget=float(RMS_BUDGET))
if STUDY_SIZE == "smoke":
    kwargs.update(train_tasks=(0,), validation_tasks=(1,), test_tasks=(2,),
                  train_starts=3, validation_starts=2, test_starts=2)
config = SteeringConfig(**kwargs)
RUN_DIR = DRIVE_OUTPUTS / "matched-steering" / RUN_NAME
print("Checkpoint:", CHECKPOINT)
print("Notebook experiment:", STEERING_NOTEBOOK_VERSION[:12])
print("Persistent output:", RUN_DIR)
print("Cases:", {s: sum(c['split'] == s for c in config.cases()) for s in ('train','validation','test')})
report = run_steering_experiment(config, CHECKPOINT, RUN_DIR,
    source_digest=STEERING_NOTEBOOK_VERSION, phase=PHASE)
if report:
    display(Markdown(f"Linear probe: **{report['dense_score']:.3f}** points. "
                     f"Sparse transcoder: **{report['sparse_score']:.3f}** points. "
                     f"Paired difference (sparse − dense): **{report['sparse_minus_dense']:.3f}**. "
                     f"95% task-bootstrap interval: {report['task_bootstrap_95ci']}."))


In [ ]:
# @title Inspect Saved Results Without Rerunning
for name in ("native_contrasts.json", "discovery.json", "frozen_selection.json", "test_progress.json", "report.json"):
    path = RUN_DIR / name
    if path.exists():
        data = json.loads(path.read_text())
        print(name)
        print(json.dumps({k:v for k,v in data.items() if k not in ("handles", "selected", "rows", "trials", "validation_trials")}, indent=2))
